In [1]:
# 整理数据
import orjsonl

In [2]:
from pathlib import Path
from extract4chem_fluorine.tools.data_tool import processDoc
import os
import re

def is_valid(s):
    return bool(re.match(r'^\d+\.?\d*$|^\d*\.\d+$', s))
input_data = {
    "experiment" : "",
    "result and discussion" : "",
}

inpath = r"/Data/home/ywwang/code/extract4chem_fluorine/data/out/20251109.json"
raw_text_path = Path("/mnt/samba_shared/wyw/data/氟基高分子_20251109183100")
import random
for record in random.sample(orjsonl.load(inpath),10):
    # 读取对应的文本
    raw_text = (raw_text_path / record["file_name"]).read_text(encoding="utf-8")
    splited_texts = processDoc(raw_text)
    
    index = 0
    exp_infos = []
    result_infos = []
    while index<=len(splited_texts)-1:
        if "REFERENCE" in splited_texts[index]["content_title"].upper():
            ...
        else:
            exp_infos.append(f"# {splited_texts[index]["content_title"]}\n{splited_texts[index]["content"]}")
        input_data["experiment"] = "\n\n".join(exp_infos)
        
        
        
        if any(f in splited_texts[index]["content_title"].upper() for f in ["RESULTS", "DISCUSSION", "CONCLUSION"]):
            raw_result_title = splited_texts[index]["content_title"]
            result_infos.append(f"# {splited_texts[index]['content_title']}\n{splited_texts[index]['content']}")
            for block in splited_texts[index+1:]:        
                if any(f in block["content_title"].lower() for f in ["conclusion", "acknowledgement", "reference"]):
                    break
                result_infos.append(f"# {block['content_title']}\n{block['content']}")
        input_data["result and discussion"] = "\n\n".join(result_infos)    
        index += 1
    break
# 这里的输出数据input_data没问题
# 下一步对各个部分进行各属性的抽取
need_cols = ["id", "aliases"]

syn_input = {
    "paper_info": input_data["experiment"],
    "material_info": {k:v  for k,v in record["main_extract_result"]["parse"]["material_ids"][0].items() if k in need_cols}
}
    


In [3]:
input_data

{'experiment': "# Enhancing Li-ion conduction and mechanical properties via addition of fluorine-containing metal-organic frameworks in all-solid-state cross-linked hyperbranched polymer electrolytes\nWen Wen $^{1,2,\\S}$ , Qinghui Zeng $^{1,\\S}$ , Pingping Chen $^{1}$ , Xin Wen $^{1}$ , Zhenfeng Li $^{1}$ , Yu Liu $^{1}$ , Jiazhu Guan $^{1}$ , Anqi Chen $^{1}$ , Wei Liu $^{1}$  ( $\\boxtimes$ ), and Liaoyun Zhang $^{1}$  ( $\\boxtimes$ )  \n$^{1}$  School of chemical sciences, University of Chinese Academy of Sciences, Beijing 100049, China\n$^{2}$  Research Institute of Petroleum Exploration & Development (RIPED), PetroChina, Beijing 100083, China\n$\\S$  Wen Wen and Qinghui Zeng contributed equally to this work.  \n© Tsinghua University Press 2022  \nReceived: 10 March 2022 / Revised: 4 May 2022 / Accepted: 9 May 2022\n\n# ABSTRACT\nPoly(ethylene oxide) (PEO)-based solid polymer electrolytes (SPEs) are commonly used in lithium metal batteries (LMBs) for their good Li-salt solvating

In [15]:
splited_texts[:10]

[{'title_level': 'H1',
  'content_title': 'Enhancing Li-ion conduction and mechanical properties via addition of fluorine-containing metal-organic frameworks in all-solid-state cross-linked hyperbranched polymer electrolytes',
  'content': 'Wen Wen $^{1,2,\\S}$ , Qinghui Zeng $^{1,\\S}$ , Pingping Chen $^{1}$ , Xin Wen $^{1}$ , Zhenfeng Li $^{1}$ , Yu Liu $^{1}$ , Jiazhu Guan $^{1}$ , Anqi Chen $^{1}$ , Wei Liu $^{1}$  ( $\\boxtimes$ ), and Liaoyun Zhang $^{1}$  ( $\\boxtimes$ )  \n$^{1}$  School of chemical sciences, University of Chinese Academy of Sciences, Beijing 100049, China\n$^{2}$  Research Institute of Petroleum Exploration & Development (RIPED), PetroChina, Beijing 100083, China\n$\\S$  Wen Wen and Qinghui Zeng contributed equally to this work.  \n© Tsinghua University Press 2022  \nReceived: 10 March 2022 / Revised: 4 May 2022 / Accepted: 9 May 2022'},
 {'title_level': 'H1',
  'content_title': 'ABSTRACT',
  'content': 'Poly(ethylene oxide) (PEO)-based solid polymer electrol

In [4]:
print(
    input_data
)

{'experiment': "# Enhancing Li-ion conduction and mechanical properties via addition of fluorine-containing metal-organic frameworks in all-solid-state cross-linked hyperbranched polymer electrolytes\nWen Wen $^{1,2,\\S}$ , Qinghui Zeng $^{1,\\S}$ , Pingping Chen $^{1}$ , Xin Wen $^{1}$ , Zhenfeng Li $^{1}$ , Yu Liu $^{1}$ , Jiazhu Guan $^{1}$ , Anqi Chen $^{1}$ , Wei Liu $^{1}$  ( $\\boxtimes$ ), and Liaoyun Zhang $^{1}$  ( $\\boxtimes$ )  \n$^{1}$  School of chemical sciences, University of Chinese Academy of Sciences, Beijing 100049, China\n$^{2}$  Research Institute of Petroleum Exploration & Development (RIPED), PetroChina, Beijing 100083, China\n$\\S$  Wen Wen and Qinghui Zeng contributed equally to this work.  \n© Tsinghua University Press 2022  \nReceived: 10 March 2022 / Revised: 4 May 2022 / Accepted: 9 May 2022\n\n# ABSTRACT\nPoly(ethylene oxide) (PEO)-based solid polymer electrolytes (SPEs) are commonly used in lithium metal batteries (LMBs) for their good Li-salt solvating

In [5]:
from langchain_core.prompts import ChatPromptTemplate
prompt_temp = ChatPromptTemplate.from_template(Path("../prompts/synthesis.txt").read_text(encoding="utf-8"))
prompt_temp.pretty_print()

================================ Human Message =================================

<System>
你是“合成信息抽取器（synthesis extractor）”。输入：一个样品及其相关信息、以及若干论文片段。只做一件事：
—— 从这些片段中抽取仅与 <MATERIAL_ID> 相关的合成信息，并输出严格 JSON。

边界：
- 仅使用提供文本；不得臆测；宁缺毋滥。
- 只抽“合成”相关内容（配方/原料/结晶/后处理/负载等）；表征/性能忽略。
- 句式如 “all/each sample …” 视为适用于 <MATERIAL_ID>。
- 本步不识别框架码；

填充规则：
- 单位统一：温度→°C；结晶时间→day；后处理时间→hour；金属负载→wt%（数值放在 token 文本中亦可）。
- 不能确定即填 null；不要猜。
- 若同一字段在多处冲突，选择与 <MATERIAL_ID> 关联最强、语义最明确的一处；仍不确定则置 null。
- 严格 JSON：只输出上述对象；双引号；true/false/null；无尾随逗号；不得输出解释性文字。

{schema_define}
</System>

<User>
<MATERIAL>
{material_info}
</MATERIAL>

<Paper>
{paper_info}
</Paper>
</User>


In [6]:
from extract4chem_fluorine.tools import MyJSONParser
from extract4chem_fluorine.entities.synthesis import SynthesisResult
syn_parser = MyJSONParser(SynthesisResult)
print(syn_parser.get_format_instructions()[:500])

严格按照以下schema输出**合法json**，除此之外绝不得输出任何其他文字:
<SCHEMA>
{
  "$defs": {
    "Crystallization": {
      "description": "晶化条件：沸石晶体生长阶段的核心参数。\n- 温度单位固定为 °C（数值）\n- 时间单位固定为 day（数值）\n- 搅拌速度单位固定为 rpm（数值）",
      "properties": {
        "temp_c": {
          "anyOf": [
            {
              "type": "number"
            },
            {
              "type": "null"
            }
          ],
          "default": null,
          "description": "晶化生长的温度（°C，数值）",
          "title": "Temp C"
        },
     


In [7]:
from extract4chem_fluorine.llm_config import llm_manager
main_llm = llm_manager["qwen_stream"]

/Data/home/ywwang/code/extract4chem_fluorine/src/extract4chem_fluorine/llm_config.py:65: UserWarning: WARNING! stream is not default parameter.
                stream was transferred to model_kwargs.
                Please confirm that stream is what you intended.
  self._models[model_name] = init_chat_model(**self._configs[model_name])


In [9]:
for x in main_llm.stream("你是谁，用json简短回答"):
    if x.content.strip() == "":
        continue
    print(x.content, end="\n", flush=True)

```json
{

  "name": "Q
wen",
  "role":
 "AI Assistant",
 
 "description": "A
 large language model developed by
 Tongyi Lab."

}
```


In [10]:
prompt_temp.input_variables

['material_info', 'paper_info', 'schema_define']

In [11]:
main_chain = prompt_temp.partial(schema_define=syn_parser.get_format_instructions()) | main_llm  | syn_parser

In [12]:
syn_input

{'paper_info': "# Enhancing Li-ion conduction and mechanical properties via addition of fluorine-containing metal-organic frameworks in all-solid-state cross-linked hyperbranched polymer electrolytes\nWen Wen $^{1,2,\\S}$ , Qinghui Zeng $^{1,\\S}$ , Pingping Chen $^{1}$ , Xin Wen $^{1}$ , Zhenfeng Li $^{1}$ , Yu Liu $^{1}$ , Jiazhu Guan $^{1}$ , Anqi Chen $^{1}$ , Wei Liu $^{1}$  ( $\\boxtimes$ ), and Liaoyun Zhang $^{1}$  ( $\\boxtimes$ )  \n$^{1}$  School of chemical sciences, University of Chinese Academy of Sciences, Beijing 100049, China\n$^{2}$  Research Institute of Petroleum Exploration & Development (RIPED), PetroChina, Beijing 100083, China\n$\\S$  Wen Wen and Qinghui Zeng contributed equally to this work.  \n© Tsinghua University Press 2022  \nReceived: 10 March 2022 / Revised: 4 May 2022 / Accepted: 9 May 2022\n\n# ABSTRACT\nPoly(ethylene oxide) (PEO)-based solid polymer electrolytes (SPEs) are commonly used in lithium metal batteries (LMBs) for their good Li-salt solvating

In [13]:
results = []
for x in main_chain.stream(syn_input):
    results.append(x)

In [14]:
import json
print(json.dumps(
    results[-1].model_dump(mode = "json"),
    indent=2,
    ensure_ascii=False
    ))

{
  "parse": {
    "material_id": "UiO-66-(F)4",
    "synthesis": {
      "zeolite_type": null,
      "gel_composition": "5.2 mmol ZrOCl2·8H2O : 5 mmol H2fBDC in 50 mL water/acetic acid (3:2 v/v)",
      "template": null,
      "silica_source": null,
      "aluminium_source": null,
      "crystallization": {
        "temp_c": 80.0,
        "time_d": 1.0,
        "agitation_rpm": null
      },
      "post_treatment_steps": [
        "washed_methanol",
        "dried_120c_24h"
      ]
    },
    "why": "From section 2.1.2: Synthesis of UiO-66-(F)4 used ZrOCl2·8H2O (5.2 mmol) and H2fBDC (5 mmol) suspended in water/acetic acid (3:2 v/v), heated at 80°C under reflux for 24 h (crystallization time converted to 1 day). Post-treatment: washed with methanol for 3 days (token 'washed_methanol'), dried at 120°C under vacuum for 24 h (token 'dried_120c_24h'). No template, silica source, or aluminium source mentioned; UiO-66-(F)4 is a MOF, not a zeolite."
  },
  "raw_text": "{\n  \"material_id\": \